# 02 · 因子研究

**目的**:在数据 + 因子模块之上,系统评估 305 个因子:

1. 每个因子的横截面 **IC**(对 7 日收益标签)
2. **Lasso(ElasticNet L1 路径)** 筛选结果
3. **随机森林 variable importance** 排名(LightGBM `boosting_type='rf'`,真 bootstrap 森林、原生处理 NaN)

⚠️ 管线的既有教训(README 有完整消融):单变量 IC 和 Lasso 用于 selection 头都**伤害**策略,
生产配置是「策展池直入」。本 notebook 是研究工具,不是生产配置。

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent                      # launched from research/
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
import binance_data as bd
import factors as F
import models as M

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)
plt.rcParams['figure.figsize'] = (13, 5)
print(f"project root: {ROOT}")

## 1. 构建因子面板(约 10-20 秒)

In [ ]:
frames = bd.load_universe()
panel = F.build_panel(frames, verbose=True)
feats = F.feature_columns(panel)
TARGET = 'target_ret_7d'
df = panel.reset_index().dropna(subset=[TARGET])
print(f"{len(feats)} features | {len(df):,} rows")

## 2. 每个因子的 IC(日度横截面 Spearman,约 30-60 秒)

In [ ]:
daily_ic = M.daily_feature_ic(df, feats, TARGET)

ic = pd.DataFrame({'mean_ic': daily_ic.mean(), 'std_ic': daily_ic.std(),
                   'n_days': daily_ic.notna().sum()})
ic['t_stat'] = ic['mean_ic'] / (ic['std_ic'] / np.sqrt(ic['n_days']))
ic = ic.sort_values('mean_ic')
print(f"|IC|>=0.03: {(ic['mean_ic'].abs() >= 0.03).sum()}/{len(ic)}  |  "
      f"|t|>=2: {(ic['t_stat'].abs() >= 2).sum()}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
top = pd.concat([ic.head(15), ic.tail(15)])
colors = ['crimson' if v < 0 else 'seagreen' for v in top['mean_ic']]
ax1.barh(top.index, top['mean_ic'], color=colors)
ax1.set_title(f'IC 两端各 15 (target={TARGET})'); ax1.tick_params(labelsize=8)
ax1.axvline(0, color='k', lw=0.6)
ax2.hist(ic['mean_ic'], bins=60, color='steelblue')
ax2.axvline(0.03, color='crimson', ls='--'); ax2.axvline(-0.03, color='crimson', ls='--')
ax2.set_title('305 个因子的 IC 分布 (虚线=±0.03)')
plt.tight_layout(); plt.show()

### 按族聚合的 IC(用 compute_factors 的插入顺序确定族边界)

In [ ]:
single = F.compute_factors(frames[config.BENCHMARK_SYMBOL])
sfeats = [c for c in single.columns if c not in F.FACTOR_PREFIX_SKIP]
bounds = [('A 动量', 'ret_1d'), ('B 波动', 'vol_5d'), ('C 形态', 'hl_range_5d'),
          ('D 趋势', 'sma_ratio_5d'), ('E 振荡', 'rsi_7d'), ('F 量能', 'volume_z_5d'),
          ('G 主动买卖', 'taker_imb_1d'), ('H 衍生品', 'funding_1d'), ('日历', 'dow_sin')]
idx = [sfeats.index(b[1]) for b in bounds] + [len(sfeats)]
fam_of = {}
for i, (fam, _) in enumerate(bounds):
    for f in sfeats[idx[i]:idx[i+1]]:
        fam_of[f] = fam
ic['family'] = [fam_of.get(f, 'I 横截面') for f in ic.index]

fam = ic.groupby('family').agg(n=('mean_ic', 'size'),
                               mean_abs_ic=('mean_ic', lambda s: s.abs().mean()),
                               best=('mean_ic', lambda s: s.abs().max()),
                               n_above_003=('mean_ic', lambda s: (s.abs() >= 0.03).sum()))
fam.sort_values('mean_abs_ic', ascending=False)

## 3. Lasso(L1 路径)筛选

`cross_sectional=True` 先按日对 X 和 y 做截面标准化——路径选的是解释**横截面排序**的特征,市场水平项被去掉。(约 1-2 分钟)

In [ ]:
lasso_cs = M.select_features_lasso(df, feats, TARGET, cross_sectional=True)
lasso_raw = M.select_features_lasso(df, feats, TARGET, cross_sectional=False)
ic_set = set(ic[ic['mean_ic'].abs() >= 0.03].index)

print(f"lasso(截面): {len(lasso_cs)} 个 | lasso(原始): {len(lasso_raw)} 个 | |IC|>=0.03: {len(ic_set)} 个")
print(f"lasso截面 ∩ IC: {len(set(lasso_cs) & ic_set)}")
print(f"lasso截面 ∩ lasso原始: {len(set(lasso_cs) & set(lasso_raw))}")
print()
print("lasso(截面) 选中,按族:")
sel_fam = pd.Series([fam_of.get(f, 'I 横截面') for f in lasso_cs]).value_counts()
print(sel_fam.to_string())

## 4. 随机森林 variable importance

LightGBM 的 `boosting_type='rf'` 是真正的 bootstrap 随机森林(树独立、行列抽样),且原生处理 NaN。用生产的策展池(排除有害的 A/B/D/F 新增族)。(约 1-2 分钟)

In [ ]:
from lightgbm import LGBMRegressor

pool = [c for c in feats if c not in set(F.SELECTION_EXCLUDE)]
rf = LGBMRegressor(boosting_type='rf', n_estimators=400, num_leaves=63,
                   max_depth=8, min_child_samples=50,
                   bagging_fraction=0.7, bagging_freq=1, feature_fraction=0.5,
                   random_state=42, n_jobs=-1, verbosity=-1)
rf.fit(df[pool], df[TARGET])

rf_imp = pd.Series(rf.booster_.feature_importance('gain'), index=pool)
rf_imp = rf_imp / rf_imp.sum()
top30 = rf_imp.nlargest(30)

fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(top30.index[::-1], top30.values[::-1], color='darkorange')
ax.set_title('随机森林 gain importance top 30 (策展池 236 特征)')
ax.tick_params(labelsize=8)
plt.tight_layout(); plt.show()

### 三种方法的对比:top-30 交集

In [ ]:
rank_ic = ic['mean_ic'].abs().nlargest(30).index
rank_rf = rf_imp.nlargest(30).index
rank_lasso = pd.Index(lasso_cs)

comp = pd.DataFrame({
    '方法': ['|IC| top30', 'RF importance top30', f'lasso截面({len(lasso_cs)})'],
    'IC∩': [30, len(set(rank_rf) & set(rank_ic)), len(set(rank_lasso) & set(rank_ic))],
    'RF∩': [len(set(rank_ic) & set(rank_rf)), 30, len(set(rank_lasso) & set(rank_rf))],
})
display(comp)

both = sorted(set(rank_ic) & set(rank_rf))
print(f"IC 与 RF 都认可的因子({len(both)} 个):")
print(both)

## 结论区(手写)

- 哪些族/因子被三种方法一致认可:
- 单变量 IC 的高分区被什么主导(参考:波动率块):
- 想尝试加入/删除的因子:

**下一步**:任何筛选想法都要过 `models.walk_forward`(每折训练窗内重选)的检验,
再看 `03_backtest_research` 的策略层结果——管线已两次证明「模型层更好 ≠ 策略层更好」。